In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_until_next_meeting AS (
-- SOURCE DECISION (re-proven Nov 12 2026 with fresh data, I re-examined): the
-- spec source "CRM activities, TYPE = Meeting, MEETING_STARTS_AT >= CURRENT_DATE"
-- cannot produce upcoming meetings. CRM Meetings are Zoom-style call logs written
-- post-hoc: 18,458 of 20,109 logged at/after their start; silver ingested through
-- 11:01 UTC today and ZERO activities of ANY type carry a future starts_at — the
-- only >= CURRENT_DATE rows are same-day calls already completed. Spec-literal =
-- the original "zero results". Calendly is the only usable source of upcoming
-- meetings; matched through silver.user_identity_map (2_8 identity spine):
-- invitee 32-hex hash OR real email against spine identity keys (pre-flight
-- validated Nov 1: 464 clients vs 434 via inline bridges). Historical footnote:
-- 241 of 20,109 CRM Meetings were pre-created >2 days ahead (max 336d), so the
-- CRM CAN carry future rows in principle — but has none in any current snapshot.
-- BOUNDARY FIX (Nov 12 2026): spec boundary is >= CURRENT_DATE — same-day
-- meetings qualify as 0 days. Was DATE(EVENT_START_TIME) > CURRENT_TIMESTAMP
-- (excluded every same-day meeting); now DATE(EVENT_START_TIME) >= CURRENT_DATE.
-- Dry-run Nov 12: 399 CC leads, 101 same-day (0 days), min 0 max 78.
-- PICK (deliberate interpretation): MIN(EVENT_START_TIME) per client = the NEXT
-- meeting (metric name days_until_next_meeting). The spec's "latest qualifying
-- meeting start" read literally = MAX = the FARTHEST-out meeting (78 days when
-- one is tomorrow) — not taken; flip MIN to MAX if truly intended.
-- Dedup for duplicate EVENT_URI per spec: INVITEE_UPDATED_AT, then
-- EVENT_CREATED_AT, then bronze_insert_date.
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
calendly_future AS (
    SELECT
        INVITEE_NAME,
        INVITEE_EMAIL,
        EVENT_START_TIME
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE EVENT_START_TIME IS NOT NULL
      AND DATE(EVENT_START_TIME) >= CURRENT_DATE
    -- Spec dedup for duplicate EVENT_URI: INVITEE_UPDATED_AT, then EVENT_CREATED_AT, then insert date
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC, EVENT_CREATED_AT DESC, bronze_insert_date DESC) = 1
),
calendly_matched AS (
    SELECT lead_id, MIN(EVENT_START_TIME) AS next_start
    FROM (
        SELECT DISTINCT
            explode(sp.lead_ids) AS lead_id,
            cf.EVENT_START_TIME
        FROM calendly_future cf
        JOIN crm_ingestion.silver.user_identity_map sp
          ON (cf.INVITEE_NAME IS NOT NULL AND array_contains(sp.user_32digits, LOWER(cf.INVITEE_NAME)))
          OR (cf.INVITEE_EMAIL IS NOT NULL AND array_contains(sp.real_emails, LOWER(cf.INVITEE_EMAIL)))
    )
    GROUP BY lead_id
)
SELECT
    cc.lead_id,
    DATEDIFF(DATE(cm.next_start), CURRENT_DATE) AS days_until_next_meeting
FROM coaching_clients cc
JOIN calendly_matched cm ON cc.lead_id = cm.lead_id
)

# Verification

Checks that the rebuilt table is populated and the values are sane.

In [0]:
display(spark.sql("""
SELECT COUNT(*) AS clients_with_next_meeting,
       COUNT(DISTINCT lead_id) AS distinct_clients,
       MIN(days_until_next_meeting) AS soonest_days,
       MAX(days_until_next_meeting) AS farthest_days,
       ROUND(AVG(days_until_next_meeting), 1) AS avg_days
FROM crm_ingestion.gold.days_until_next_meeting
"""))

display(spark.sql("""
SELECT CASE WHEN days_until_next_meeting <= 7 THEN 'within a week'
            WHEN days_until_next_meeting <= 30 THEN 'within a month'
            WHEN days_until_next_meeting <= 90 THEN 'within a quarter'
            ELSE 'beyond 90 days' END AS booking_window,
       COUNT(*) AS clients
FROM crm_ingestion.gold.days_until_next_meeting
GROUP BY 1 ORDER BY 1
"""))